# 02 Retrieval Baseline

Inputs: output dataset from notebook `01_build_25d_cache.ipynb`.

This notebook trains a lightweight image-text retrieval baseline from
the 2.5D cache. It reports closed-pool Recall@K and train-index
retrieved-report metrics. No val/test report is placed in the train
retrieval index for report generation.

Gate tối thiểu: Recall@5 >= 0.20, MRR >= 0.08, retrieved ROUGE-L >= 0.18,
clinical keyword F1 >= 0.25.

In [1]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_similarity

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
RET_DIR = OUTPUT_ROOT / "02_retrieval"
RET_DIR.mkdir(parents=True, exist_ok=True)

def locate_cache():
    roots = list(Path("/kaggle/input").rglob("q1_cache_meta.json")) if Path("/kaggle/input").exists() else []
    roots += list(Path.cwd().rglob("q1_cache_meta.json"))
    if not roots: raise FileNotFoundError("q1_cache_meta.json not found. Add notebook 01 output.")
    meta_path = sorted(roots, key=lambda p: len(str(p)))[0]
    return meta_path.parent, json.loads(meta_path.read_text())
CACHE_DIR, meta = locate_cache()
clean = pd.read_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv")
cache_index = pd.read_csv(CACHE_DIR / "q1_cache_index.csv")
valid = cache_index[cache_index.cache_ok].copy()
clean = clean.drop(columns=["row_index"], errors="ignore").merge(valid[["case_id", "row_index"]], on="case_id", how="inner")
target_col = meta.get("target_column", "target_text")
if target_col in clean.columns:
    clean["target_text"] = clean[target_col].fillna("").astype(str)
elif "report_text_clean" in clean.columns:
    clean["target_text"] = clean["report_text_clean"].fillna("").astype(str)
else:
    clean["target_text"] = clean["target_text"].fillna("").astype(str)
arr = np.memmap(CACHE_DIR / meta["mmap_path"], dtype=np.uint8, mode="r", shape=(meta["n"], meta["channels"], meta["image_size"], meta["image_size"]))
print("Loaded cache:", CACHE_DIR, "usable rows:", len(clean))

Loaded cache: /kaggle/input/notebooks/hannhu4002/01-streaming-2-5d-cache-builder/vimedpet_q1_outputs/01_cache usable rows: 2757


In [2]:
import re, math, json
from collections import Counter

KEYWORDS = ["hạch", "phổi", "xương", "gan", "thận", "lách", "đại tràng", "SUV", "FDG", "tổn thương", "viêm", "PET/CT"]

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x or "").strip())

def lcs_len(a, b):
    dp = [0] * (len(b) + 1)
    for ca in a:
        prev = 0
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev + 1 if ca == cb else max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

def rouge_l_f1(pred, ref):
    pred_toks = normalize_text(pred).split()
    ref_toks = normalize_text(ref).split()
    if not pred_toks or not ref_toks:
        return 0.0
    lcs = lcs_len(pred_toks, ref_toks)
    p = lcs / len(pred_toks)
    r = lcs / len(ref_toks)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def bleu4_simple(pred, ref):
    pred_toks = normalize_text(pred).split()
    ref_toks = normalize_text(ref).split()
    if len(pred_toks) < 4 or len(ref_toks) < 4:
        return 0.0
    scores = []
    for n in range(1, 5):
        pred_ngrams = Counter(tuple(pred_toks[i:i+n]) for i in range(len(pred_toks)-n+1))
        ref_ngrams = Counter(tuple(ref_toks[i:i+n]) for i in range(len(ref_toks)-n+1))
        overlap = sum((pred_ngrams & ref_ngrams).values())
        total = max(1, sum(pred_ngrams.values()))
        scores.append((overlap + 1) / (total + 1))
    bp = min(1.0, math.exp(1 - len(ref_toks) / max(1, len(pred_toks))))
    return float(bp * math.exp(sum(math.log(s) for s in scores) / 4))

def keyword_f1(pred, ref, keywords=KEYWORDS):
    pred_set = {kw.lower() for kw in keywords if kw.lower() in normalize_text(pred).lower()}
    ref_set = {kw.lower() for kw in keywords if kw.lower() in normalize_text(ref).lower()}
    if not pred_set and not ref_set:
        return 1.0
    tp = len(pred_set & ref_set)
    p = tp / max(1, len(pred_set))
    r = tp / max(1, len(ref_set))
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def extract_suv_values(text):
    values = []
    for m in re.finditer(r"SUV(?:max)?\s*[:=]?\s*([0-9]+(?:[\\.,][0-9]+)?)", str(text), flags=re.I):
        try:
            values.append(float(m.group(1).replace(",", ".")))
        except Exception:
            pass
    return values

def suv_tolerance_accuracy(pred, ref, tol=0.5):
    pv = extract_suv_values(pred)
    rv = extract_suv_values(ref)
    if not pv and not rv:
        return 1.0
    if not pv or not rv:
        return 0.0
    used = set()
    correct = 0
    for x in pv:
        candidates = [(abs(x - y), j) for j, y in enumerate(rv) if j not in used]
        if not candidates:
            continue
        err, j = min(candidates)
        if err <= tol:
            correct += 1
            used.add(j)
    return correct / max(len(rv), len(pv), 1)

def section_validity(text):
    t = normalize_text(text).lower()
    return int(("finding" in t or "mô tả" in t or "kết quả" in t) and ("impression" in t or "kết luận" in t))

def evaluate_prediction_frame(df, pred_col="prediction", ref_col="target_text"):
    rows = []
    for _, row in df.iterrows():
        pred, ref = row[pred_col], row[ref_col]
        rows.append({
            "rouge_l": rouge_l_f1(pred, ref),
            "bleu4": bleu4_simple(pred, ref),
            "keyword_f1": keyword_f1(pred, ref),
            "suv_acc_0_5": suv_tolerance_accuracy(pred, ref, tol=0.5),
            "section_valid": section_validity(pred),
        })
    out = pd.DataFrame(rows)
    return out.mean(numeric_only=True).to_dict()

In [3]:
def image_features(row_indices, out_size=32):
    feats = []
    step = meta["image_size"] // out_size
    for idx in row_indices:
        x = arr[int(idx)].astype(np.float32) / 255.0
        x = x[:, :out_size*step, :out_size*step].reshape(meta["channels"], out_size, step, out_size, step).mean(axis=(2,4))
        feats.append(x.reshape(-1))
    return np.asarray(feats, dtype=np.float32)

texts = clean["target_text"].fillna("").astype(str).tolist()
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
X_text = tfidf.fit_transform(texts)
n_comp = min(256, X_text.shape[1] - 1, len(clean) - 1)
text_svd = TruncatedSVD(n_components=max(8, n_comp), random_state=391)
T = normalize(text_svd.fit_transform(X_text))

X_img = image_features(clean["row_index"].values)
scaler = StandardScaler(with_mean=True, with_std=True)
X_img = scaler.fit_transform(X_img)
img_svd = TruncatedSVD(n_components=min(T.shape[1], X_img.shape[1]-1), random_state=391)
I0 = normalize(img_svd.fit_transform(X_img))

train_mask = clean.clean_split == "train"
reg = Ridge(alpha=10.0)
reg.fit(I0[train_mask], T[train_mask])
I = normalize(reg.predict(I0))
T = normalize(T[:, :I.shape[1]])

In [4]:
def recall_mrr_for_split(split):
    idx = np.where(clean.clean_split.values == split)[0]
    sims = cosine_similarity(I[idx], T[idx])
    ranks = []
    for i in range(len(idx)):
        order = np.argsort(-sims[i])
        rank = int(np.where(order == i)[0][0]) + 1
        ranks.append(rank)
    return {
        f"{split}_closed_recall@1": float(np.mean([r <= 1 for r in ranks])),
        f"{split}_closed_recall@5": float(np.mean([r <= 5 for r in ranks])),
        f"{split}_closed_mrr": float(np.mean([1/r for r in ranks])),
    }

metrics = {}
for split in ["validation", "test"]:
    if (clean.clean_split == split).any():
        metrics.update(recall_mrr_for_split(split))

train_idx = np.where(train_mask.values)[0]
train_text_emb = T[train_idx]
pred_rows = []
for split in ["validation", "test"]:
    q_idx = np.where(clean.clean_split.values == split)[0]
    sims = cosine_similarity(I[q_idx], train_text_emb)
    best = sims.argmax(axis=1)
    for local_i, train_best in enumerate(best):
        qi = q_idx[local_i]
        ti = train_idx[train_best]
        pred_rows.append({
            "split": split,
            "case_id": clean.iloc[qi].case_id,
            "retrieved_train_case_id": clean.iloc[ti].case_id,
            "prediction": clean.iloc[ti].target_text,
            "target_text": clean.iloc[qi].target_text,
        })
pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(RET_DIR / "retrieval_predictions_val_test.csv", index=False, encoding="utf-8-sig")
gen_metrics = evaluate_prediction_frame(pred_df)
metrics.update({f"retrieved_{k}": float(v) for k, v in gen_metrics.items()})

(RET_DIR / "retrieval_metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metrics, indent=2, ensure_ascii=False))
display(pred_df.head(5))

{
  "validation_closed_recall@1": 0.002421307506053269,
  "validation_closed_recall@5": 0.014527845036319613,
  "validation_closed_mrr": 0.01698898394426606,
  "test_closed_recall@1": 0.0,
  "test_closed_recall@5": 0.012106537530266344,
  "test_closed_mrr": 0.014318780311659333,
  "retrieved_rouge_l": 0.7793286064944692,
  "retrieved_bleu4": 0.6609676786681993,
  "retrieved_keyword_f1": 0.8749567623659634,
  "retrieved_suv_acc_0_5": 0.5980629539951574,
  "retrieved_section_valid": 0.0
}


,split,case_id,retrieved_train_case_id,prediction,target_text
0,validation,1ddcef3193ae,6620bc775264,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...
1,validation,2df72b4b39a5,6620bc775264,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...
2,validation,b08e7dbaea56,6620bc775264,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...
3,validation,f4067447144b,d9a0e3ad6857,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...
4,validation,dbdaab9d8c2b,6620bc775264,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...,Hình ảnh bắt xạ theo đặc điểm sinh lý ở não. T...
